# CUDA QLoRA Fine-Tuning — Veritas Verifier (Kaggle/Colab)

This notebook fine-tunes `TinyLlama/TinyLlama-1.1B-Chat-v1.0` with **4-bit QLoRA** (bitsandbytes + peft) on the same Verdict/Explanation/Citation fact-verification task used by the Apple Silicon MLX LoRA adapter (`checkpoints/mlx_lora_verifier/`).

It requires a **CUDA GPU** (Kaggle: Settings > Accelerator > GPU T4 x2, or Colab: Runtime > Change runtime type > GPU). It will not run on a CPU/MPS-only machine.

Run the cells in order:
1. Check GPU availability.
2. Install dependencies.
3. Clone the Veritas repo.
4. Export the flattened SFT dataset (`data/processed/sft_train.jsonl`, `sft_val.jsonl`).
5. Train the QLoRA adapter and evaluate it.
6. Zip the resulting artifacts and download them.

**Outputs to copy back into the repo** (see the final cell):
- `checkpoints/cuda_qlora_verifier/` (LoRA adapter files)
- `reports/cuda_qlora_eval.json` / `reports/cuda_qlora_eval.md`
- `data/processed/sft_train.jsonl` / `data/processed/sft_val.jsonl`

Per project rules, CUDA QLoRA is only considered complete once these files exist from an actual run of this notebook — do not fabricate them.

## 1. Check GPU availability

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "This notebook requires a CUDA GPU. On Kaggle: Settings > Accelerator > "
        "GPU T4 x2. On Colab: Runtime > Change runtime type > GPU."
    )

## 2. Install dependencies

In [ ]:
!pip install -q -U torch transformers peft bitsandbytes accelerate datasets trl

## 3. Clone the Veritas repo

In [ ]:
import os

if not os.path.exists("Veritas"):
    !git clone https://github.com/sushildalavi/veritas.git Veritas
%cd Veritas

## 4. Export the flattened SFT dataset

This step works on CPU and is a quick sanity check that the training data (`data/processed/mlx_lora/{train,valid}.jsonl`) loads and flattens correctly before spending GPU time.

In [ ]:
!python3 scripts/train_cuda_qlora.py --export-sft-only

## 5. Train the CUDA QLoRA adapter and evaluate it

Trains on `data/processed/mlx_lora/train.jsonl` (600 examples), then evaluates on `data/processed/mlx_lora/valid.jsonl` (100 examples) using the same metrics as the MLX LoRA evaluation: verdict accuracy, macro F1, per-class F1, citation valid rate, unsupported-sentence rate, and mean latency.

In [ ]:
!python3 scripts/train_cuda_qlora.py \
    --base-model TinyLlama/TinyLlama-1.1B-Chat-v1.0 \
    --output-dir checkpoints/cuda_qlora_verifier \
    --report-json reports/cuda_qlora_eval.json \
    --report-md reports/cuda_qlora_eval.md

In [ ]:
import json

with open("reports/cuda_qlora_eval.json") as handle:
    print(json.dumps(json.load(handle), indent=2))

## 6. Zip artifacts and download

In [ ]:
!zip -r cuda_qlora_artifacts.zip \
    checkpoints/cuda_qlora_verifier \
    reports/cuda_qlora_eval.json \
    reports/cuda_qlora_eval.md \
    data/processed/sft_train.jsonl \
    data/processed/sft_val.jsonl

In [ ]:
try:
    from google.colab import files
    files.download("cuda_qlora_artifacts.zip")
except ImportError:
    print(
        "Not running in Colab. On Kaggle, find cuda_qlora_artifacts.zip in "
        "/kaggle/working/Veritas and download it from the notebook's Output tab."
    )

## Files to copy back into the Veritas repo

After unzipping `cuda_qlora_artifacts.zip`, copy these paths into your local clone of the repo (preserving the directory structure):

- `checkpoints/cuda_qlora_verifier/` → `checkpoints/cuda_qlora_verifier/`
- `reports/cuda_qlora_eval.json` → `reports/cuda_qlora_eval.json`
- `reports/cuda_qlora_eval.md` → `reports/cuda_qlora_eval.md`
- `data/processed/sft_train.jsonl` → `data/processed/sft_train.jsonl`
- `data/processed/sft_val.jsonl` → `data/processed/sft_val.jsonl`

Once `checkpoints/cuda_qlora_verifier/` and `reports/cuda_qlora_eval.json` exist in the repo, CUDA QLoRA can be marked complete, and `checkpoints/cuda_qlora_verifier/` becomes the base adapter for CUDA DPO (`notebooks/13_cuda_dpo_kaggle_colab.ipynb`).